## סדרה מדודה: קצב שינוי ואירועים

סדרה עיתית היא מערך של זמנים ומערך של ערכים שנמדדו בהם; לכמה ריצות, מערך בצורה (ריצה, זמן), כמו בשבוע 9.

קצב השינוי מתקבל מההפרשים בין ערכים סמוכים, ו-n מדידות נותנות n-1 מרווחים, כמו בלולאות של החצי הראשון. במערך, ‎`x[1:] - x[:-1]`‎ מחסר מכל ערך את הערך שלפניו, ו-‎`np.diff(x)`‎ הוא קיצור לאותה פעולה. כל הפרש שייך למרווח שבין שתי מדידות, ולכן הזמן שלו הוא אמצע המרווח. בתא הבא כדור מתגלגל במורד מדרון, ומיקומו נמדד כל חצי שנייה:

```{image} ../../figures/10_1_diff_intervals.svg
:alt: Four points bound three intervals; each value of np.diff sits at the middle of its interval
:align: center
```

In [ ]:
import numpy as np

t = np.array([0.0, 0.5, 1.0, 1.5])      # time (s)
x = np.array([0.0, 0.25, 1.0, 2.25])    # position (m)
print(x[1:] - x[:-1])                   # each value minus the one before
print(np.diff(x))                       # the same, with np.diff
v = np.diff(x) / np.diff(t)             # velocity in each interval (m/s)
t_mid = (t[:-1] + t[1:]) / 2            # the middle of each interval (s)
print(v)
print(t_mid)

כאן $x = t^2$ ו-$v = 2t$: המהירות של כל מרווח שווה בדיוק למהירות באמצע המרווח, ולא בקצותיו.

### דוגמה: שלוש זריקות שנמדדו בסרטון

שלוש זריקות של כדור, בזווית 45 מעלות, צולמו בסרטון של 20 תמונות בשנייה, ובכל תמונה נמדד מיקום הכדור במטרים (‎`t`‎, ‎`x`‎ ו-‎`y`‎). הכדור נזרק מגג, מגובה היד, והגובה נמדד מגובה הגג, ולכן בסוף הסרטון הגבהים שליליים. בתא הבא המהירות האנכית בכל מרווח, ומההפרש השני התאוצה, לאורך ציר הזמן, ‎`axis=1`‎.

In [ ]:
import numpy as np

# "measured" positions (m) of three throws, from a video at 20 frames/s
rng = np.random.default_rng(6)
v0_true = np.array([19.0, 20.0, 21.0])       # launch speeds (m/s)
angle = np.radians(45)
t = np.round(np.arange(65) * 0.05, 2)        # frame times (s)
x = v0_true[:, None] * np.cos(angle) * t
y = 1.5 + v0_true[:, None] * np.sin(angle) * t - 4.9 * t**2
x = np.round(x + rng.normal(0, 0.003, x.shape), 3)
y = np.round(y + rng.normal(0, 0.003, y.shape), 3)

In [ ]:
print(t.shape, y.shape)           # (time,) and (run, time)
dt = t[1] - t[0]                  # time between frames (s)
t_mid = (t[:-1] + t[1:]) / 2
vy = np.diff(y, axis=1) / dt      # velocity in each interval (m/s)
ay = np.diff(vy, axis=1) / dt     # the second difference (m/s^2)
print(vy.shape, ay.shape)
print(ay.mean(axis=1))            # one mean acceleration per run

In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(1, 2, figsize=(10, 4))
for i in range(3):
    axs[0].plot(t, y[i], "o", markersize=3, label=f"Run {i + 1}")
    axs[1].plot(t_mid, vy[i], "o", markersize=3, label=f"Run {i + 1}")
axs[0].set_xlabel("t (s)")
axs[0].set_ylabel("y (m)")
axs[0].set_title("Measured height")
axs[1].set_xlabel("t (s)")
axs[1].set_ylabel(r"$v_y$ (m/s)")
axs[1].set_title("Velocity from np.diff, at the interval midpoints")
axs[0].legend()
plt.show()

כל הפרש מקצר את ציר הזמן באחד: 65 מדידות, 64 מהירויות, 63 תאוצות. תאוצה בודדת רועשת, כי הפרשים מגדילים את אי-הדיוק של המדידה, אבל הממוצע בכל ריצה קרוב ל-$-g = -9.8$ מ'/ש².

### אירועים: השיא והמעבר אל מתחת לגג

השיא הוא האינדקס של הגובה הגדול ביותר, ‎`argmax`‎ לאורך ציר הזמן, והזמן שלו הוא האינדקס כפול ‎`dt`‎, כי הסרטון מתחיל ב-0. הרגע שבו הכדור עובר אל מתחת לגג הוא הזמן הראשון שבו ‎`y < 0`‎: מסכה על הזמנים, כמו רגע הנחיתה בשבוע 7, לכל ריצה בנפרד.

In [ ]:
i_peak = y.argmax(axis=1)         # the highest frame of each run
print(i_peak, i_peak * dt)        # its index and its time (s)

for i in range(3):
    t_below = t[y[i] < 0]         # the times when run i is below the roof
    print(t_below[0])             # the first of them (s)

בריצה השלישית, הגבוהה ביותר, שני האירועים מאוחרים יותר; הדיוק הוא תמונה אחת, 0.05 שניות.

### דוגמה נוספת: עגלה מתנודדת על מסילת אוויר

עגלה על מסילת אוויר, קשורה לקפיץ, הוסטה 8 ס"מ מנקודת שיווי המשקל, ‎`x = 0`‎, ושוחררה. חיישן תנועה מדד את מיקומה 50 פעמים בשנייה, במשך 5 שניות (‎`t_glider`‎ ו-‎`x_glider`‎, במטרים). את המחזור מוצאים מהרגעים שבהם העגלה עוברת דרך 0: לשתי מדידות סמוכות עם סימנים שונים יש מכפלה שלילית, ולכן המסכה ‎`x[:-1] * x[1:] < 0`‎ מסמנת את המרווחים האלה, והזמן האמצעי שלהם הוא רגע המעבר. בין שני מעברים עובר חצי מחזור.

In [ ]:
# "measured" position (m) of a glider on an air track, 50 readings/s
rng = np.random.default_rng(5)
t_glider = np.round(np.arange(251) * 0.02, 2)
x_glider = 0.08 * np.cos(5.0 * t_glider)     # 8 cm, omega = 5 1/s
x_glider = np.round(x_glider + rng.normal(0, 0.0002, t_glider.shape), 4)

In [ ]:
t_mid_glider = (t_glider[:-1] + t_glider[1:]) / 2
cross = x_glider[:-1] * x_glider[1:] < 0    # True where the sign changes
t_cross = t_mid_glider[cross]               # times of passing through 0
print(t_cross)
half = np.diff(t_cross)                     # half a period each (s)
print(half)
print(round(2 * half.mean(), 3))            # the period (s)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(t_glider, x_glider * 100, color="tab:blue")
ax.plot(t_cross, 0 * t_cross, "o", color="tab:red",
        label="Passing through 0")
ax.axhline(0, color="gray", linewidth=0.8)
ax.set_xlabel("t (s)")
ax.set_ylabel("x (cm)")
ax.legend()
plt.show()

המחזור שנמדד, 1.257 שניות, מתאים לנוסחה $T = 2\pi\sqrt{m/k}$ עם מסת העגלה, 0.2 ק"ג, וקבוע הקפיץ, 5 ניוטון למטר. כל רגע מעבר ידוע רק עד כדי מרווח אחד, ולכן כל הפרש הוא 0.62 או 0.64, והממוצע מדויק יותר.

### באג נפוץ: אורך diff קצר באחד

‎`np.diff(y, axis=1)`‎ על מערך בצורה ‎`(3, 65)`‎ מחזיר מערך בצורה ‎`(3, 64)`‎. ציור של ריצה אחת מול ‎`t`‎ המקורי, שאורכו 65, נעצר בשגיאה:

```text
ValueError: x and y must have same first dimension, but have shapes (65,) and (64,)
```

גרוע מזה: אם משתמשים בהיסח הדעת ב-‎`t[:-1]`‎, האורכים מתאימים ואין שגיאה, אבל ציר הזמן מוזז בחצי צעד ביחס למה שהמהירות מייצגת. ‎`t_mid`‎ שבדוגמה הוא הפתרון הנכון.

### נסו בעצמכם

עגלה משוחררת ממנוחה על מסילה משופעת בזווית של 5 מעלות, וחיישן מודד את מהירותה כל 0.1 שניות (‎`t_cart`‎ ו-‎`v_cart`‎). בלי חיכוך, תאוצתה $g\sin\alpha$. השלימו את הפונקציה ‎`acceleration`‎, שמקבלת מהירויות ואת זמני המדידה ומחזירה את התאוצה $a\approx\Delta v/\Delta t$ בכל מרווח. בדקו אותה על הדוגמה שבתא, ואחר כך השוו את ממוצע התאוצות של העגלה ל-$g\sin\alpha$.

In [ ]:
# "measured" velocity (m/s) of a cart on a 5 degree slope, every 0.1 s
rng = np.random.default_rng(0)
t_cart = np.round(np.arange(21) * 0.1, 1)
v_cart = 9.8 * np.sin(np.radians(5)) * t_cart
v_cart = np.round(v_cart + rng.normal(0, 0.005, t_cart.shape), 3)
v_cart[0] = 0.0                              # released from rest

In [ ]:
def acceleration(v, t):
    """Approximate acceleration (m/s^2) in each interval: one fewer than v."""
    pass   # replace with your code

v_test = np.array([10.0, 9.0, 8.0])
t_test = np.array([0.0, 0.1, 0.2])
print(acceleration(v_test, t_test))
# expected: [-10. -10.]

`````{admonition} פתרון
:class: dropdown, tip
```python
def acceleration(v, t):
    """Approximate acceleration (m/s^2) in each interval: one fewer than v."""
    return np.diff(v) / np.diff(t)

print(acceleration(v_test, t_test))            # [-10. -10.]
a_cart = acceleration(v_cart, t_cart)
print(round(a_cart.mean(), 3))                 # 0.854
print(round(9.8 * np.sin(np.radians(5)), 3))   # 0.854
```
שלוש מהירויות נותנות שני מרווחים, ולכן שתי תאוצות. לעגלה, 21 מדידות נותנות 20 תאוצות, והממוצע שלהן, 0.854 מ'/ש², שווה ל-$g\sin\alpha$ עד הספרה השלישית.
`````

### בדקו את עצמכם

שתי השאלות מתייחסות לקוד הבא:

```python
a = np.array([1.0, 4.0, 9.0, 16.0])
print(np.diff(a))                   # line A

y = np.array([[0.0, 3.0, 4.0, 3.0, 0.0],
              [0.0, 2.0, 2.5, 2.8, 2.0]])   # (run, time)
print(y.argmax(axis=1))             # line B
```

In [ ]:
import json
from jupyterquiz import display_quiz

example = \
'''
[
    {
        "question": "כמה מספרים תדפיס שורה A?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "3",
                "correct": true,
                "feedback": "נכון: בין 4 ערכים יש 3 הפרשים: 3, 5 ו-7."
            },
            {
                "answer": "4",
                "correct": false,
                "feedback": "לא: diff מחזיר את ההפרשים בין ערכים סמוכים, ויש אחד פחות מהם."
            },
            {
                "answer": "2",
                "correct": false,
                "feedback": "לא: diff מקצר את המערך באחד בלבד."
            },
            {
                "answer": "1",
                "correct": false,
                "feedback": "לא: הפרש בודד בין הערך הראשון לאחרון לא מחושב כאן."
            }
        ]
    },
    {
        "question": "מהו המספר השני ששורה B תדפיס?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "3",
                "correct": true,
                "feedback": "נכון: בריצה השנייה הגובה הגדול ביותר, 2.8, נמצא באינדקס 3."
            },
            {
                "answer": "2",
                "correct": false,
                "feedback": "לא: 2 הוא האינדקס של השיא בריצה הראשונה, המספר הראשון בפלט."
            },
            {
                "answer": "2.8",
                "correct": false,
                "feedback": "לא: 2.8 הוא הגובה בשיא; argmax מחזיר את האינדקס שלו."
            },
            {
                "answer": "4",
                "correct": false,
                "feedback": "לא: באינדקס 4 נמצא הערך האחרון, 2.0, שאינו הגדול ביותר."
            }
        ]
    }
]
'''
myquiz = json.loads(example)
display_quiz(myquiz)

### תרגול עצמי

מתי מהירות העגלה המתנודדת גדולה ביותר, ומה גודלה? חשבו את המהירות בכל מרווח מתוך ‎`x_glider`‎, ומצאו את הגודל המרבי שלה ואת הזמן שבו הוא מתקבל.

In [ ]:
# dt_glider = ...
# v_glider = ...
# speed = abs(v_glider)
# ...

`````{admonition} פתרון
:class: dropdown, tip
```python
dt_glider = t_glider[1] - t_glider[0]
v_glider = np.diff(x_glider) / dt_glider
speed = abs(v_glider)
i = speed.argmax()
print(round(t_mid_glider[i], 2), round(speed.max(), 3))   # 4.09 0.425
```
המהירות הגדולה ביותר, 0.425 מ'/ש', מתקבלת ב-4.09 שניות, אחד מרגעי המעבר דרך 0: העגלה מהירה ביותר בנקודת שיווי המשקל. בתנועה הרמונית המהירות המרבית היא $A\omega = 0.08 \cdot 5 = 0.4$ מ'/ש'; ההפרש נובע מאי-הדיוק של המדידה.
`````